# Notebook A — Race Discovery
Search RunSignUp for races by location (city/state or zip) and/or race name.  
Output is a reviewable table and a saved CSV of candidate races with their race IDs.  
Use this to identify race IDs to pass into Notebook B.

In [80]:
import os
import requests
import pandas as pd
from pathlib import Path
from datetime import date
from dotenv import load_dotenv

load_dotenv()

OUTPUT_DIR = Path('outputs/race_discovery')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [81]:
API_KEY      = "TkPIUG2GLK95zjIAONwAyv348fQQ6TtY"
API_SECRET   = "oNsju6QeN0UEVZ1JK612wyay0dC2jLJ9"
AFLT_TOKEN   = os.environ['RUNSIGNUP_AFLT_TOKEN']

# --- Search inputs (fill in what you know, leave the rest as '') ---

RACE_NAME   = ''           # partial name ok, e.g. 'Gulf Coast' or 'Mississippi'

CITY        = 'Biloxi'     # city name
STATE       = 'MS'         # 2-letter state code
ZIPCODE     = ''           # zip code (alternative to city/state)
RADIUS_MI   = 50           # miles around zip (only used if ZIPCODE is set)

DISTANCE    = 'marathon'   # filter events to a specific distance, or '' for all
EVENT_NAME  = ''           # partial event name filter, e.g. 'Full' or 'Half', or '' for all

# Date window — only return races with events starting within this range
START_DATE_MIN = '2010-01-01'                        # YYYY-MM-DD
START_DATE_MAX = date.today().strftime('%Y-%m-%d')   # past races only

MAX_RESULTS = 50    # cap on how many races to return (max 500 per page)

# RunSignUp API base URL
BASE_URL = 'https://runsignup.com/Rest/races'

In [82]:
# --- Search inputs (fill in what you know, leave the rest as '') ---

RACE_NAME   = 'Watermelon'           # partial name ok, e.g. 'Gulf Coast' or 'Mississippi'
EVENT_NAME  = 'RUN'                   # partial name ok, e.g. 'Marathon' or 'Half Marathon'       

CITY        = 'Jackson'     # city name
STATE       = 'MS'         # 2-letter state code
ZIPCODE     = ''           # zip code (alternative to city/state)
RADIUS_MI   = 50           # miles around zip (only used if ZIPCODE is set)

DISTANCE    = '5K'   # filter to a specific distance, or '' for all

# Date window — only return races with events starting on or after this date
# Set to a year far back to capture historical races
START_DATE_MIN = '2010-01-01'   # YYYY-MM-DD
START_DATE_MAX = ''             # YYYY-MM-DD or '' for no upper limit

MAX_RESULTS = 50    # cap on how many races to return (max 500 per page)

# RunSignUp API base URL
BASE_URL = 'https://runsignup.com/Rest/races'

In [83]:
params = {
    'format':             'json',
    'results_per_page':   MAX_RESULTS,
    'page':               1,
    'include_event_days': 'T',
    'api_key':            API_KEY,
    'api_secret':         API_SECRET,
    'aflt_token':         AFLT_TOKEN,
}

if RACE_NAME:       params['name']           = RACE_NAME
if CITY:            params['city']           = CITY
if STATE:           params['state']          = STATE
if ZIPCODE:         params['zipcode']        = ZIPCODE
if ZIPCODE:         params['radius']         = RADIUS_MI
if START_DATE_MIN:  params['start_date_min'] = START_DATE_MIN
if START_DATE_MAX:  params['start_date_max'] = START_DATE_MAX

print('Calling RunSignUp API...')
print(f'Parameters: {params}\n')

response = requests.get(BASE_URL, params=params)
response.raise_for_status()
data = response.json()

raw_races = data.get('races', [])
print(f'Races returned: {len(raw_races)}')

Calling RunSignUp API...
Parameters: {'format': 'json', 'results_per_page': 50, 'page': 1, 'include_event_days': 'T', 'api_key': 'TkPIUG2GLK95zjIAONwAyv348fQQ6TtY', 'api_secret': 'oNsju6QeN0UEVZ1JK612wyay0dC2jLJ9', 'aflt_token': 'uL3LJsNhaU6SFxIRWfJFjA0K24o2fddK', 'name': 'Watermelon', 'city': 'Jackson', 'state': 'MS', 'start_date_min': '2010-01-01'}

Races returned: 1


## Parse Results into a Flat Table

In [84]:
rows = []

for item in raw_races:
    race = item.get('race', {})

    race_id   = race.get('race_id', '')
    race_name = race.get('name', '')
    race_url  = race.get('url', '')

    address   = race.get('address', {})
    city      = address.get('city', '')
    state     = address.get('state', '')
    zipcode   = address.get('zipcode', '')

    next_date = race.get('next_date', '')

    # Pull event-level info if present
    events = race.get('events', [])
    if events:
        for event in events:
            rows.append({
                'race_id':        race_id,
                'race_name':      race_name,
                'event_id':       event.get('event_id', ''),
                'event_name':     event.get('name', ''),
                'distance':       event.get('distance', ''),
                'start_date':     event.get('start_time', ''),
                'city':           city,
                'state':          state,
                'zipcode':        zipcode,
                'race_url':       race_url,
            })
    else:
        # No event detail — just log the race-level info
        rows.append({
            'race_id':        race_id,
            'race_name':      race_name,
            'event_id':       '',
            'event_name':     '',
            'distance':       '',
            'start_date':     next_date,
            'city':           city,
            'state':          state,
            'zipcode':        zipcode,
            'race_url':       race_url,
        })

df = pd.DataFrame(rows, columns=[
    'race_id', 'race_name', 'event_id', 'event_name',
    'distance', 'start_date', 'city', 'state', 'zipcode', 'race_url'
])

print(f'Total rows parsed: {len(df)}')
df

Total rows parsed: 1


,race_id,race_name,event_id,event_name,distance,start_date,city,state,zipcode,race_url
0,32802,Farm Bureau Watermelon Classic 5k Run/Walk,,,,07/04/2026,Jackson,MS,39216,https://runsignup.com/Race/MS/Jackson/Watermel...


## Fetch Child Events for Each Discovered Race
A race ID represents the series; each year's running is a child event. This cell calls
`/Rest/race/{race_id}` for every race found above to retrieve the full event list.

In [85]:
event_rows = []

for race_id in df['race_id'].unique():
    resp = requests.get(
        f'https://runsignup.com/Rest/race/{race_id}',
        params={
            'format':          'json',
            'include_events':  'T',
            'api_key':         API_KEY,
            'api_secret':      API_SECRET,
            'aflt_token':      AFLT_TOKEN,
        }
    )
    resp.raise_for_status()
    race = resp.json().get('race', {})
    race_name = race.get('name', '')

    for event in race.get('events', []):
        event_rows.append({
            'race_id':    race_id,
            'race_name':  race_name,
            'event_id':   event.get('event_id', ''),
            'event_name': event.get('name', ''),
            'distance':   event.get('distance', ''),
            'start_time': event.get('start_time', ''),
        })

df_events = pd.DataFrame(event_rows, columns=[
    'race_id', 'race_name', 'event_id', 'event_name', 'distance', 'start_time'
])

if DISTANCE:
    df_events = df_events[df_events['distance'] == DISTANCE]
if EVENT_NAME:
    df_events = df_events[df_events['event_name'].str.contains(EVENT_NAME, case=False, na=False)]

print(f'Total events across all races: {len(df_events)}')
df_events

Total events across all races: 11


,race_id,race_name,event_id,event_name,distance,start_time
0,32802,Farm Bureau Watermelon Classic 5k Run/Walk,1155721,2026 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2026 07:30
3,32802,Farm Bureau Watermelon Classic 5k Run/Walk,987494,2025 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2025 07:30
6,32802,Farm Bureau Watermelon Classic 5k Run/Walk,843361,2024 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2024 07:30
9,32802,Farm Bureau Watermelon Classic 5k Run/Walk,698223,2023 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2023 07:30
12,32802,Farm Bureau Watermelon Classic 5k Run/Walk,609109,2022 Farm Bureau Watermelon Classic 5k RUN,5K,7/4/2022 07:30
15,32802,Farm Bureau Watermelon Classic 5k Run/Walk,495705,2021 Farm Bureau Watermelon Classic 5k RUN,5K,7/3/2021 07:30
18,32802,Farm Bureau Watermelon Classic 5k Run/Walk,407416,2020 Farm Bureau Watermelon Classic 5k Run,5K,7/4/2020 00:00
20,32802,Farm Bureau Watermelon Classic 5k Run/Walk,308814,2019 Farm Bureau Watermelon Classic 5k Run,5K,7/4/2019 07:30
23,32802,Farm Bureau Watermelon Classic 5k Run/Walk,233263,2018 Farm Bureau Watermelon Classic 5k Run,5K,7/4/2018 07:30
26,32802,Farm Bureau Watermelon Classic 5k Run/Walk,147717,2017 Farm Bureau Watermelon Classic 5k Run,5K,7/4/2017 07:30


## Inspect Raw JSON for Any Race
If you want to see everything the API returned for a specific race — useful for finding fields not captured above.

In [86]:
import json

# Set to the index position in raw_races you want to inspect (0 = first result)
INSPECT_INDEX = 0

if raw_races:
    print(json.dumps(raw_races[INSPECT_INDEX], indent=2))
else:
    print('No results to inspect.')

{
  "race": {
    "race_id": 32802,
    "name": "Farm Bureau Watermelon Classic 5k Run/Walk",
    "last_date": "07/04/2025",
    "last_end_date": "07/04/2025",
    "next_date": "07/04/2026",
    "next_end_date": "07/04/2026",
    "is_draft_race": "F",
    "is_private_race": "F",
    "is_registration_open": "T",
    "created": "4/4/2016 14:52",
    "last_modified": "5/5/2026 15:02",
    "description": "<p>The Annual Farm Bureau Watermelon Classic benefitting the Mississippi Sports Hall of Fame &amp; Museum will take place on Saturday, July 4, 2026! This annual event is one of the largest fundraisers for the Museum each year and helps keep the doors open to visitors locally and all across the world to achieve our mission of preserving, protecting and promote Mississippi&#39;s rich sports heritage for this and future generations. Race day events will include a competitive 5k Run, 5k Walk, and a One Mile Fun Run.\u00a0</p>\n<p>\u00a0</p>",
    "url": "https://runsignup.com/Race/MS/Jackson/

## Save Results to CSV

In [87]:
today = date.today().strftime('%Y%m%d')
filename = OUTPUT_DIR / f'race_discovery_{today}.csv'

df_events.to_csv(filename, index=False)
print(f'Saved: {filename}')
print(f'Rows: {len(df_events)}')

Saved: outputs\race_discovery\race_discovery_20260509.csv
Rows: 11
